In [ ]:
# 14_rag_eval — RAG quality evaluation (retrieval / citation / groundedness)
# For each eval-set case (data/dev/rag/rag_eval_set.jsonl), retrieve + generate, then load the metrics into rag.eval_results.
# retrieval recall / citation rate are measured even without an LLM.
# groundedness / correctness are filled via Databricks Agent Evaluation (mlflow.evaluate) when an LLM endpoint is available.
# eval is an isolated layer the detection pipeline doesn't read. It's not added to the main pipeline job.


In [ ]:
# Databricks Runtime 15.4 LTS may not include the Vector Search SDK by default.
# Keep this as a notebook-level guard for manual runs; job tasks also declare the PyPI library.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("databricks.vector_search") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "databricks-vectorsearch"])


In [ ]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("vs_endpoint", "access_drift_vs")
dbutils.widgets.text("index_name", "")
dbutils.widgets.text("llm_endpoint", "databricks-claude-sonnet-5")
dbutils.widgets.text("top_k", "4")
dbutils.widgets.text("rag_eval_path", "data/dev/rag/rag_eval_set.jsonl")
dbutils.widgets.text("run_id", "manual")

CATALOG = dbutils.widgets.get("catalog")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
INDEX_NAME = dbutils.widgets.get("index_name") or f"{CATALOG}.rag.doc_chunks_index"
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
TOP_K = int(dbutils.widgets.get("top_k"))
EVAL_RELATIVE_PATH = dbutils.widgets.get("rag_eval_path").strip("/")
RUN_ID = dbutils.widgets.get("run_id") or "manual"
EVAL_LOCAL_PATH = f"/Workspace/Shared/access-drift/files/{EVAL_RELATIVE_PATH}"
TARGET_TABLE = f"{CATALOG}.rag.eval_results"
GROUNDEDNESS_TARGET = 0.9


In [ ]:
import os, json

if not os.path.isfile(EVAL_LOCAL_PATH):
    raise RuntimeError(
        f"RAG eval set file not found: {EVAL_LOCAL_PATH}\n"
        "Check that the bundle deploy completed."
    )
with open(EVAL_LOCAL_PATH) as f:
    eval_cases = [json.loads(line) for line in f if line.strip()]
print(f"eval cases: {len(eval_cases)}")


In [ ]:
from databricks.vector_search.client import VectorSearchClient
from mlflow.deployments import get_deploy_client

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
deploy_client = get_deploy_client("databricks")

LLM_AVAILABLE = False
if LLM_ENDPOINT:
    try:
        deploy_client.get_endpoint(LLM_ENDPOINT)
        LLM_AVAILABLE = True
    except Exception as exc:  # noqa: BLE001
        print(f"LLM endpoint '{LLM_ENDPOINT}' unavailable -> evaluate retrieval/citation only: {exc}")

SYSTEM_PROMPT = (
    "You are a security-remediation assistant. Answer using only the content in the provided evidence documents, "
    "and append a [chunk_id] citation for the evidence at the end of each sentence. "
    "If you cannot answer from the evidence, reply only 'Insufficient evidence'."
)


def llm_payload(messages: list[dict], max_tokens: int) -> dict:
    payload = {"messages": messages, "max_tokens": max_tokens}
    # Databricks Claude/Anthropic endpoints reject temperature, while OpenAI/Llama accept it.
    if "claude" not in LLM_ENDPOINT.lower() and "anthropic" not in LLM_ENDPOINT.lower():
        payload["temperature"] = 0.0
    return payload


def retrieve(case: dict) -> list:
    filters = {
        "drift_type": case["drift_type"],
        "principal_kind": case.get("principal_kind", "nhi"),
    }
    if case.get("source_system"):
        filters["source_system"] = case["source_system"]
    res = index.similarity_search(
        query_text=case["question"],
        columns=["chunk_id", "doc_id", "source_title", "chunk_text"],
        filters=filters,
        num_results=TOP_K,
    )
    return res.get("result", {}).get("data_array") or []


def deterministic_eval_answer(docs: list, reason: str) -> str:
    if not docs:
        return "Insufficient evidence"
    return f"({reason}: generation skipped)"


def generate(case: dict, docs: list):
    if not docs:
        return deterministic_eval_answer(docs, "no search results"), "deterministic_fallback"
    if not LLM_AVAILABLE:
        return deterministic_eval_answer(docs, "LLM unavailable"), "deterministic_fallback"

    context = "\n\n".join(f"[{d[0]}] {d[3]}" for d in docs)
    try:
        resp = deploy_client.predict(
            endpoint=LLM_ENDPOINT,
            inputs=llm_payload([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Evidence:\n{context}\n\nQuestion:\n{case['question']}"},
            ], max_tokens=512),
        )
        answer = resp["choices"][0]["message"]["content"].strip()
    except Exception as exc:  # noqa: BLE001
        return f"(LLM call failed: {type(exc).__name__})", "deterministic_fallback"

    retrieved_chunk_ids = [d[0] for d in docs]
    has_citation = any(f"[{cid}]" in answer for cid in retrieved_chunk_ids)
    if not has_citation:
        return deterministic_eval_answer(docs, "no citation"), "deterministic_fallback"
    return answer, "rag"


In [ ]:
rows = []
eval_records = []  # for Agent Evaluation input
for case in eval_cases:
    docs = retrieve(case)
    retrieved_doc_ids = [d[1] for d in docs]
    retrieved_chunk_ids = [d[0] for d in docs]
    expected = case["expected_doc_ids"]
    hit = any(doc_id in retrieved_doc_ids for doc_id in expected)
    answer, source = generate(case, docs)
    has_citation = any(f"[{cid}]" in answer for cid in retrieved_chunk_ids)
    rows.append({
        "eval_case_id": case["eval_case_id"],
        "drift_type": case["drift_type"],
        "question": case["question"],
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "expected_doc_ids": expected,
        "retrieval_hit": bool(hit),
        "has_citation": bool(has_citation),
        "answer_source": source,
        "answer_text": answer,
    })
    eval_records.append({
        "request": {"messages": [{"role": "user", "content": case["question"]}]},
        "response": answer,
        "retrieved_context": [{"content": d[3], "doc_uri": d[0]} for d in docs],
        "expected_response": "\n".join(case.get("expected_facts", [])),
    })

recall = sum(r["retrieval_hit"] for r in rows) / len(rows)
citation_rate = sum(r["has_citation"] for r in rows) / len(rows)
print(f"retrieval recall@{TOP_K}: {recall:.3f}")
print(f"citation rate: {citation_rate:.3f}")


In [ ]:
# Databricks Agent Evaluation: groundedness / correctness (LLM judge).
# Only attempt when an LLM endpoint is available; even on failure, keep the retrieval/citation metrics as-is.
groundedness_by_case = {}
correctness_by_case = {}
mean_groundedness = None
mean_correctness = None

if LLM_AVAILABLE:
    try:
        import mlflow
        import pandas as pd

        eval_df = pd.DataFrame(eval_records)
        with mlflow.start_run(run_name=f"rag_eval_{RUN_ID}"):
            eval_result = mlflow.evaluate(
                data=eval_df,
                model_type="databricks-agent",
            )
        metrics = eval_result.metrics
        mean_groundedness = metrics.get("groundedness/mean") or metrics.get(
            "response/llm_judged/groundedness/mean")
        mean_correctness = metrics.get("correctness/mean") or metrics.get(
            "response/llm_judged/correctness/mean")
        per_case = eval_result.tables.get("eval_results")
        if per_case is not None:
            for i, case in enumerate(eval_cases):
                if i < len(per_case):
                    row = per_case.iloc[i]
                    groundedness_by_case[case["eval_case_id"]] = row.get("groundedness")
                    correctness_by_case[case["eval_case_id"]] = row.get("correctness")
        print(f"groundedness(mean): {mean_groundedness} | correctness(mean): {mean_correctness}")
    except Exception as exc:  # noqa: BLE001
        print(f"Agent Evaluation skipped (unavailable): {type(exc).__name__}: {exc}")
else:
    print("LLM unavailable: groundedness/correctness are stored as null.")


# Fallback LLM judge: if Agent Evaluation (databricks-agents) is unavailable,
# use the same FM endpoint as a judge to fill groundedness/correctness (internal to Databricks, no external key needed).
if LLM_AVAILABLE and mean_groundedness is None and mean_correctness is None:
    import re as _re

    def _judge_score(prompt: str) -> float | None:
        try:
            resp = deploy_client.predict(
                endpoint=LLM_ENDPOINT,
                inputs=llm_payload([
                    {"role": "system", "content":
                     "You are a RAG quality evaluator. Output only a single number between 0.0 and 1.0. Do not write any explanation."},
                    {"role": "user", "content": prompt},
                ], max_tokens=8),
            )
            text = resp["choices"][0]["message"]["content"].strip()
            match = _re.search(r"(?:0(?:\.\d+)?|1(?:\.0+)?)", text)
            return min(max(float(match.group()), 0.0), 1.0) if match else None
        except Exception as exc:  # noqa: BLE001
            print(f"judge call failed: {type(exc).__name__}")
            return None

    g_scores, c_scores = [], []
    for case, rec in zip(eval_cases, eval_records):
        if not rec["retrieved_context"]:
            continue
        context_text = "\n".join(c["content"] for c in rec["retrieved_context"])
        g = _judge_score(
            "Evaluate whether the following answer used only content in the evidence documents. "
            "The more content not in the evidence is mixed in, the lower the score.\n"
            f"Evidence:\n{context_text}\n\nAnswer:\n{rec['response']}\n\nScore (0.0-1.0):")
        c = _judge_score(
            "Evaluate how accurately the following answer captures the expected facts.\n"
            f"Expected facts:\n{rec['expected_response']}\n\nAnswer:\n{rec['response']}\n\nScore (0.0-1.0):")
        if g is not None:
            groundedness_by_case[case["eval_case_id"]] = g
            g_scores.append(g)
        if c is not None:
            correctness_by_case[case["eval_case_id"]] = c
            c_scores.append(c)
    mean_groundedness = sum(g_scores) / len(g_scores) if g_scores else None
    mean_correctness = sum(c_scores) / len(c_scores) if c_scores else None
    print(f"fallback LLM judge({LLM_ENDPOINT}) — groundedness(mean): {mean_groundedness} | correctness(mean): {mean_correctness}")


In [ ]:
from pyspark.sql import types as T
from datetime import datetime, timezone
import json

now = datetime.now(timezone.utc)
records = []
for r in rows:
    records.append({
        "eval_case_id": r["eval_case_id"],
        "drift_type": r["drift_type"],
        "question": r["question"],
        "retrieved_chunk_ids": json.dumps(r["retrieved_chunk_ids"], ensure_ascii=False),
        "expected_doc_ids": json.dumps(r["expected_doc_ids"], ensure_ascii=False),
        "retrieval_hit": r["retrieval_hit"],
        "has_citation": r["has_citation"],
        "answer_source": r["answer_source"],
        "groundedness": (float(groundedness_by_case[r["eval_case_id"]])
                         if groundedness_by_case.get(r["eval_case_id"]) is not None else None),
        "correctness": (float(correctness_by_case[r["eval_case_id"]])
                        if correctness_by_case.get(r["eval_case_id"]) is not None else None),
        "answer_text": r["answer_text"],
        "llm_endpoint": LLM_ENDPOINT if LLM_AVAILABLE else "none",
        "evaluated_at": now,
        "run_id": RUN_ID,
    })

schema = T.StructType([
    T.StructField("eval_case_id", T.StringType(), False),
    T.StructField("drift_type", T.StringType(), False),
    T.StructField("question", T.StringType(), False),
    T.StructField("retrieved_chunk_ids", T.StringType(), False),
    T.StructField("expected_doc_ids", T.StringType(), False),
    T.StructField("retrieval_hit", T.BooleanType(), False),
    T.StructField("has_citation", T.BooleanType(), False),
    T.StructField("answer_source", T.StringType(), False),
    T.StructField("groundedness", T.DoubleType(), True),
    T.StructField("correctness", T.DoubleType(), True),
    T.StructField("answer_text", T.StringType(), False),
    T.StructField("llm_endpoint", T.StringType(), False),
    T.StructField("evaluated_at", T.TimestampType(), False),
    T.StructField("run_id", T.StringType(), False),
])
target_columns = spark.table(TARGET_TABLE).columns
spark.createDataFrame(records, schema=schema).select(*target_columns) \
    .write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)

summary = [
    "rag eval passed",
    f"cases: {len(rows)}",
    f"retrieval_recall@{TOP_K}: {recall:.3f}",
    f"citation_rate: {citation_rate:.3f}",
    f"groundedness_mean: {mean_groundedness} (target >= {GROUNDEDNESS_TARGET})",
    f"correctness_mean: {mean_correctness}",
    f"llm_available: {LLM_AVAILABLE}",
]
dbutils.notebook.exit("\n".join(summary))
